In [ ]:
import os

# Voir la structure exacte de votre dataset Kaggle
for root, dirs, files in os.walk('/kaggle/input'):
    # Afficher seulement les 3 premiers niveaux
    level = root.replace('/kaggle/input', '').count(os.sep)
    if level < 3:
        indent = '  ' * level
        print(f'{indent}📁 {os.path.basename(root)}/')
        if level == 2:
            xml_files = [f for f in files if f.endswith('.xml')]
            img_files = [f for f in files if f.lower().endswith(('.jpg','.jpeg','.png'))]
            txt_files = [f for f in files if f.endswith('.txt')]
            if xml_files:  print(f'{indent}   📄 {len(xml_files)} XML')
            if img_files:  print(f'{indent}   🖼️  {len(img_files)} images')
            if txt_files:  print(f'{indent}   📝 {len(txt_files)} TXT')

In [ ]:
import os
import re
import shutil
import xml.etree.ElementTree as ET
from pathlib import Path

# ════════════════════════════════════════════
# CHEMINS KAGGLE
# ════════════════════════════════════════════
INPUT_ROOT  = '/kaggle/input/datasets/amirabbes03/dataset'
OUTPUT_ROOT = '/kaggle/working/dataset_yolo'
SPLITS      = ['train', 'val', 'test']
EXTENSIONS_IMG = ('.jpg', '.jpeg', '.png', '.bmp', '.tiff')

CLASSES = [
    'voilier', 'yacht', 'jet_ski', 'bateau_peche',
    'navire_croisiere', 'navire_militaire', 'remorqueur', 'cargo',
]
CLASS_TO_ID = {cls: idx for idx, cls in enumerate(CLASSES)}

# ════════════════════════════════════════════
# FONCTIONS
# ════════════════════════════════════════════
def polygon_to_bbox(points_str, img_w, img_h):
    try:
        coords = [tuple(map(float, pt.split(','))) for pt in points_str.strip().split(';')]
        xs = [p[0] for p in coords]
        ys = [p[1] for p in coords]
        x_centre = (min(xs)+max(xs)) / 2 / img_w
        y_centre = (min(ys)+max(ys)) / 2 / img_h
        largeur  = (max(xs)-min(xs)) / img_w
        hauteur  = (max(ys)-min(ys)) / img_h
        return (
            max(0.0, min(1.0, x_centre)),
            max(0.0, min(1.0, y_centre)),
            max(0.0, min(1.0, largeur)),
            max(0.0, min(1.0, hauteur)),
        )
    except:
        return None

def parse_cvat_xml(xml_path):
    tree = ET.parse(xml_path)
    root = tree.getroot()
    data = {}
    for img_elem in root.findall('image'):
        nom   = img_elem.get('name', '')
        img_w = int(img_elem.get('width',  0))
        img_h = int(img_elem.get('height', 0))
        if img_w == 0 or img_h == 0:
            continue
        annotations = []
        for poly in img_elem.findall('polygon'):
            label      = poly.get('label', '').lower().strip()
            points_str = poly.get('points', '')
            if label not in CLASS_TO_ID or not points_str:
                continue
            bbox = polygon_to_bbox(points_str, img_w, img_h)
            if bbox:
                annotations.append((CLASS_TO_ID[label], *bbox))
        data[nom] = annotations
    return data

# ════════════════════════════════════════════
# CONVERSION
# ════════════════════════════════════════════
print('🔄 Conversion CVAT XML → YOLO TXT\n')

total_images = 0
total_labels = 0
total_polys  = 0

for split in SPLITS:
    split_in  = os.path.join(INPUT_ROOT,  split)
    images_out = os.path.join(OUTPUT_ROOT, split, 'images')
    labels_out = os.path.join(OUTPUT_ROOT, split, 'labels')
    os.makedirs(images_out, exist_ok=True)
    os.makedirs(labels_out, exist_ok=True)

    if not os.path.exists(split_in):
        print(f'  ⚠️  {split} introuvable')
        continue

    # Trouver tous les batches
    batches = sorted([
        d for d in os.listdir(split_in)
        if re.match(r'^batch\w+$', d, re.IGNORECASE)
        and os.path.isdir(os.path.join(split_in, d))
    ])

    n_img = n_ann = n_poly = 0
    print(f'📁 {split.upper()} — {len(batches)} batches')

    for nom_batch in batches:
        dossier_batch = os.path.join(split_in, nom_batch)

        # Trouver le XML
        xml_path = None
        for f in os.listdir(dossier_batch):
            if f.endswith('_cvat.xml'):
                xml_path = os.path.join(dossier_batch, f)
                break

        if xml_path is None:
            print(f'  ⚠️  [{nom_batch}] pas de _cvat.xml — ignoré')
            continue

        annotations = parse_cvat_xml(xml_path)

        # Traiter les images
        images = sorted([
            f for f in os.listdir(dossier_batch)
            if f.lower().endswith(EXTENSIONS_IMG)
        ])

        for nom_image in images:
            src = os.path.join(dossier_batch, nom_image)
            dst = os.path.join(images_out, nom_image)
            shutil.copy2(src, dst)

            # Chercher annotations
            anns = annotations.get(nom_image, [])
            if not anns:
                for key in annotations:
                    if os.path.basename(key) == nom_image:
                        anns = annotations[key]
                        break

            # Écrire TXT
            stem     = Path(nom_image).stem
            txt_path = os.path.join(labels_out, f'{stem}.txt')
            with open(txt_path, 'w') as f:
                for (cls_id, xc, yc, w, h) in anns:
                    f.write(f'{cls_id} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}\n')
                    n_poly += 1

            n_img += 1
            if anns: n_ann += 1

In [ ]:
!pip install ultralytics -q
print('✅ ultralytics installé')

In [ ]:
import os

# Créer le dossier si absent
os.makedirs('/kaggle/working/dataset_yolo', exist_ok=True)

# Créer data.yaml manuellement
yaml_content = """path: /kaggle/working/dataset_yolo
train: train/images
val:   val/images
test:  test/images

nc: 8
names: ['voilier', 'yacht', 'jet_ski', 'bateau_peche',
        'navire_croisiere', 'navire_militaire', 'remorqueur', 'cargo']
"""

with open('/kaggle/working/dataset_yolo/data.yaml', 'w') as f:
    f.write(yaml_content)

print("✅ data.yaml créé !")
print(open('/kaggle/working/dataset_yolo/data.yaml').read())

for split in ['train', 'val', 'test']:
    imgs   = len(os.listdir(f'/kaggle/working/dataset_yolo/{split}/images'))
    labels = len(os.listdir(f'/kaggle/working/dataset_yolo/{split}/labels'))
    print(f'{split:6s} → images: {imgs:4d} | labels: {labels:4d}')

In [ ]:
# train Yolo11n sans augmentation de données

from ultralytics import YOLO

model = YOLO('yolo11n.pt')

model.train(
    data    = '/kaggle/working/dataset_yolo/data.yaml',
    epochs  = 150,
    imgsz   = 640,
    batch   = 16,
    device  = 0,
    project = '/kaggle/working/runs',
    name    = 'maritime_yolov11',
)

In [ ]:
from ultralytics import YOLO

# ✅ Bon chemin — maritime_yolov11 (sans le -2)
model = YOLO('/kaggle/working/runs/maritime_yolov11/weights/best.pt')

# Évaluer sur TEST
metrics = model.val(
    data   = '/kaggle/working/dataset_yolo/data.yaml',
    split  = 'test',
    device = 0,
)

print('═' * 50)
print('  📊 Résultats sur TEST set')
print('═' * 50)
print(f'  mAP50        : {metrics.box.map50:.4f}')
print(f'  mAP50-95     : {metrics.box.map:.4f}')
print(f'  Précision    : {metrics.box.mp:.4f}')
print(f'  Rappel       : {metrics.box.mr:.4f}')
print('═' * 50)

CLASSES = ['voilier','yacht','jet_ski','bateau_peche',
           'navire_croisiere','navire_militaire','remorqueur','cargo']

print('\n  📋 Par classe :')
for i, cls in enumerate(CLASSES):
    emoji = '✅' if metrics.box.ap50[i]>=0.75 else '🟠' if metrics.box.ap50[i]>=0.5 else '🔴'
    print(f'  {emoji} {cls:<25} mAP50={metrics.box.ap50[i]:.4f}')

In [ ]:
# train Yolo11n avec augmentation de données

# ============================================
# 8. ENTRAÎNEMENT YOLO11n
# ============================================

print("\n" + "="*60)
print("🚀 ENTRAÎNEMENT YOLO11 SUR SPLIT CORRIGÉ")
print("="*60)

model = YOLO('yolo11n.pt')

results = model.train(
    data=yaml_path,
    epochs=150,
    imgsz=640,
    batch=16,
    device=0,
    
    # Augmentations conservatrices
    augment=True,
    fliplr=0.3,
    degrees=5.0,
    mixup=0.05,
    copy_paste=0.0,
    
    patience=50,
    
    project='/kaggle/working/runs_stratified',
    name='maritime_stratified',
    exist_ok=True
)

print("\n" + "="*60)
print("🎉 SCRIPT TERMINÉ !")
print("="*60)
print(f"\n📍 Nouveau dataset: {OUTPUT_DIR}")
print(f"📍 Résultats: /kaggle/working/runs_stratified/maritime_stratified/")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from pathlib import Path

print("="*70)
print("📊 ANALYSE APPROFONDIE DES PERFORMANCES YOLO11n")
print("="*70)

# ============================================
# 1. CHARGER LES RÉSULTATS
# ============================================

results_path = '/kaggle/working/runs_stratified/maritime_stratified/results.csv'
df = pd.read_csv(results_path)

print(f"\n✅ Fichier résultats chargé: {len(df)} epochs")

# ============================================
# 2. ANALYSE DU SUR-APPRENTISSAGE (OVERFITTING)
# ============================================

print("\n" + "="*70)
print("🔍 1. DIAGNOSTIC DE SUR-APPRENTISSAGE")
print("="*70)

# Récupérer les dernières valeurs
last_epoch = df['epoch'].iloc[-1]
last_train_loss = df['train/box_loss'].iloc[-1]
last_val_loss = df['val/box_loss'].iloc[-1]
gap = abs(last_train_loss - last_val_loss)

print(f"\n Dernière époque: {last_epoch}")
print(f" 📉 Train loss final: {last_train_loss:.4f}")
print(f" 📉 Val loss finale:  {last_val_loss:.4f}")
print(f" 📊 Écart Train-Val:   {gap:.4f}")

# Diagnostic
if gap < 0.05:
    print(f"\n ✅ Écart très faible -> PAS de sur-apprentissage significatif")
elif gap < 0.10:
    print(f"\n ⚠️ Écart modéré -> Léger sur-apprentissage")
else:
    print(f"\n 🔴 Écart important -> Sur-apprentissage sévère")

# Évolution des pertes
train_loss_start = df['train/box_loss'].iloc[0]
val_loss_start = df['val/box_loss'].iloc[0]
reduction_train = (1 - last_train_loss/train_loss_start)*100
reduction_val = (1 - last_val_loss/val_loss_start)*100

print(f"\n 📉 Réduction train loss: {reduction_train:.1f}%")
print(f" 📉 Réduction val loss:   {reduction_val:.1f}%")

# ============================================
# 3. QUALITÉ DE LA CONVERGENCE
# ============================================

print("\n" + "="*70)
print("🎯 2. QUALITÉ DE LA CONVERGENCE")
print("="*70)

# Meilleures métriques
best_map50_idx = df['metrics/mAP50(B)'].idxmax()
best_map50 = df['metrics/mAP50(B)'].max()
best_map95 = df['metrics/mAP50-95(B)'].max()
epoch_best = df['epoch'].iloc[best_map50_idx]

print(f"\n 🏆 Meilleure mAP50:    {best_map50:.4f} à l'époque {epoch_best}")
print(f" 🏆 Meilleure mAP50-95: {best_map95:.4f}")
print(f" 🏆 Dernière mAP50:     {df['metrics/mAP50(B)'].iloc[-1]:.4f}")

# Vérifier si le modèle a encore progressé récemment
recent_improvement = df['metrics/mAP50(B)'].iloc[-10:].max() - df['metrics/mAP50(B)'].iloc[-10:].min()
print(f" 📈 Amélioration des 10 dernières époques: {recent_improvement:.4f}")

if recent_improvement < 0.01:
    print("   ⚠️ Le modèle a stagné -> Pas besoin de plus d'époques")
else:
    print("   ✅ Le modèle progresse encore -> Plus d'époques pourraient aider")

# ============================================
# 4. ANALYSE DES PERFORMANCES PAR CLASSE
# ============================================

print("\n" + "="*70)
print("📋 3. PERFORMANCES PAR CLASSE")
print("="*70)

# Données manuelles d'après votre sortie
classes = ['voilier', 'yacht', 'jet_ski', 'bateau_peche', 
           'navire_croisiere', 'navire_militaire', 'remorqueur', 'cargo']

precision = [0.522, 0.564, 1.0, 0.376, 0.461, 0.613, 0.709, 0.651]
recall = [0.235, 0.588, 0.045, 0.164, 0.79, 0.713, 0.567, 0.662]
map50 = [0.269, 0.605, 0.126, 0.201, 0.562, 0.636, 0.640, 0.647]
map95 = [0.140, 0.404, 0.063, 0.115, 0.471, 0.505, 0.431, 0.481]

print("\n Classe            | Precision | Recall  | mAP50   | mAP50-95 | Statut")
print(" " + "-"*70)

for i, cls in enumerate(classes):
    if map95[i] < 0.1:
        status = "🔴 CRITIQUE"
    elif map95[i] < 0.2:
        status = "🟠 FAIBLE"
    elif map95[i] < 0.4:
        status = "🟡 MOYEN"
    else:
        status = "🟢 BON"
    
    print(f" {cls:17} | {precision[i]:8.3f} | {recall[i]:7.3f} | {map50[i]:7.3f} | {map95[i]:8.3f} | {status}")

# Classes problématiques
problematic = [classes[i] for i in range(len(classes)) if map95[i] < 0.2]
print(f"\n 🔴 Classes problématiques (mAP95 < 0.2): {len(problematic)}")
for cls in problematic:
    print(f"    - {cls}")

# ============================================
# 5. VISUALISATIONS
# ============================================

print("\n" + "="*70)
print("📈 4. GÉNÉRATION DES GRAPHIQUES")
print("="*70)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Graphique 1: Évolution des pertes
ax = axes[0, 0]
ax.plot(df['epoch'], df['train/box_loss'], label='Train Loss', linewidth=2)
ax.plot(df['epoch'], df['val/box_loss'], label='Val Loss', linewidth=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title(f'Évolution des pertes (Écart final: {gap:.4f})')
ax.legend()
ax.grid(True, alpha=0.3)

# Graphique 2: mAP50 et mAP50-95
ax = axes[0, 1]
ax.plot(df['epoch'], df['metrics/mAP50(B)'], label='mAP50', linewidth=2, color='green')
ax.plot(df['epoch'], df['metrics/mAP50-95(B)'], label='mAP50-95', linewidth=2, color='blue')
ax.set_xlabel('Epoch')
ax.set_ylabel('mAP')
ax.set_title(f'mAP50: {best_map50:.3f} | mAP50-95: {best_map95:.3f}')
ax.legend()
ax.grid(True, alpha=0.3)

# Graphique 3: Performance par classe (mAP50-95)
ax = axes[1, 0]
colors = ['red' if x < 0.2 else 'orange' if x < 0.4 else 'green' for x in map95]
bars = ax.bar(classes, map95, color=colors)
ax.set_xlabel('Classe')
ax.set_ylabel('mAP50-95')
ax.set_title('Performance par classe (mAP50-95)')
ax.set_xticklabels(classes, rotation=45, ha='right')
ax.axhline(y=0.2, color='red', linestyle='--', label='Seuil critique (0.2)')
ax.axhline(y=0.4, color='orange', linestyle='--', label='Seuil moyen (0.4)')
ax.legend()

# Ajouter les valeurs sur les barres
for bar, val in zip(bars, map95):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
            f'{val:.3f}', ha='center', va='bottom', fontsize=9)

# Graphique 4: Precision vs Recall
ax = axes[1, 1]
colors_map = ['red' if x < 0.2 else 'orange' if x < 0.4 else 'green' for x in map95]
for i, cls in enumerate(classes):
    ax.scatter(recall[i], precision[i], s=200, color=colors_map[i], alpha=0.7)
    ax.annotate(cls, (recall[i], precision[i]), xytext=(5, 5), 
                textcoords='offset points', fontsize=9)
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Compromis Precision / Recall')
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.grid(True, alpha=0.3)
ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5)
ax.axvline(x=0.5, color='gray', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig('/kaggle/working/performance_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✅ Graphique sauvegardé: /kaggle/working/performance_analysis.png")

# ============================================
# 6. CONCLUSION ET RECOMMANDATION
# ============================================

print("\n" + "="*70)
print("🎯 5. CONCLUSION ET RECOMMANDATION")
print("="*70)

# Score de confiance pour YOLO11l
score_passez_a_large = 0
reasons_for = []
reasons_against = []

# Critère 1: mAP50-95 trop faible
if best_map95 < 0.4:
    score_passez_a_large += 2
    reasons_for.append(f"mAP50-95 trop faible ({best_map95:.3f} < 0.4)")

# Critère 2: Classes problématiques
if len(problematic) >= 2:
    score_passez_a_large += 2
    reasons_for.append(f"{len(problematic)} classes avec mAP95 < 0.2")

# Critère 3: Jet-ski très faible
if map95[2] < 0.1:
    score_passez_a_large += 2
    reasons_for.append("Performance jet_ski catastrophique (< 0.1)")

# Critère 4: Pas de sur-apprentissage
if gap < 0.1:
    score_passez_a_large += 1
    reasons_for.append("Pas de sur-apprentissage → peut supporter un modèle plus gros")

# Critère 5: Rappel global faible
if recall_mean := np.mean(recall) < 0.5:
    score_passez_a_large += 1
    reasons_for.append(f"Rappel moyen trop faible ({np.mean(recall):.3f} < 0.5)")

# Contre-indications
if gap > 0.15:
    reasons_against.append("Risque de sur-apprentissage avec un modèle plus gros")
if best_map95 > 0.5:
    reasons_against.append("Performances déjà bonnes, gain limité attendu")

print("\n📊 Score d'opportunité pour YOLO11l: {}/8".format(score_passez_a_large))

print("\n✅ RAISONS DE PASSER À YOLO11l:")
for reason in reasons_for:
    print(f"   • {reason}")

if reasons_against:
    print("\n⚠️ RAISONS DE RESTER SUR YOLO11n:")
    for reason in reasons_against:
        print(f"   • {reason}")

# Recommandation finale
print("\n" + "="*70)
if score_passez_a_large >= 5:
    print("🔴 RECOMMANDATION: PASSEZ À YOLO11l")
    print("   Les performances actuelles sont insuffisantes et le")
    print("   modèle peut supporter plus de paramètres.")
elif score_passez_a_large >= 3:
    print("🟡 RECOMMANDATION: TESTEZ YOLO11l (ou YOLO11m)")
    print("   Les résultats sont mitigés, un modèle plus gros")
    print("   pourrait améliorer les classes difficiles.")
else:
    print("🟢 RECOMMANDATION: RESTEZ SUR YOLO11n")
    print("   Les performances sont déjà bonnes ou le risque")
    print("   de sur-apprentissage est trop élevé.")
print("="*70)

# Générer un rapport texte
report = f"""
RAPPORT D'ANALYSE YOLO11n
{'='*50}

1. MÉTRIQUES GLOBALES
   - mAP50: {best_map50:.4f}
   - mAP50-95: {best_map95:.4f}
   - Écart train/val loss: {gap:.4f}

2. CLASSES PROBLÉMATIQUES
   {', '.join(problematic) if problematic else 'Aucune'}

3. SCORE OPPORTUNITÉ YOLO11l: {score_passez_a_large}/8

4. CONCLUSION: {'PASSER À YOLO11l' if score_passez_a_large >= 5 else 'TESTER YOLO11m' if score_passez_a_large >= 3 else 'GARDER YOLO11n'}
"""

print("\n📄 RAPPORT TEXTE SAUVEGARDÉ")
print(report)

# Sauvegarder le rapport
with open('/kaggle/working/analysis_report.txt', 'w') as f:
    f.write(report)

print("\n✅ Fichiers générés:")
print("   - performance_analysis.png")
print("   - analysis_report.txt")

In [ ]:

# ENTRAÎNEMENT YOLO11l avec augmentation de données


import os
import shutil
import yaml
import glob
from collections import defaultdict
from sklearn.model_selection import train_test_split
from ultralytics import YOLO
import pandas as pd

print("="*70)
print("🚀 SCRIPT COMPLET : DATASET + ENTRAÎNEMENT YOLO11l")
print("="*70)

# ============================================
# PARTIE 1 : CRÉATION DU DATASET STRATIFIÉ
# ============================================

print("\n" + "="*70)
print("📁 PARTIE 1 : CRÉATION DU DATASET STRATIFIÉ")
print("="*70)

DATASET_PATH = '/kaggle/input/datasets/amirabbes03/dataset'
OUTPUT_DIR = '/kaggle/working/dataset_stratified'

CLASSES = [
    'voilier', 'yacht', 'jet_ski', 'bateau_peche',
    'navire_croisiere', 'navire_militaire', 'remorqueur', 'cargo'
]
CLASS_TO_ID = {cls: idx for idx, cls in enumerate(CLASSES)}

def load_all_images(base_path):
    """Charge toutes les images avec leurs labels depuis train/labels/"""
    images_data = []
    labels_dir = os.path.join(base_path, 'train', 'labels')
    
    if not os.path.exists(labels_dir):
        print(f"❌ Labels non trouvés: {labels_dir}")
        return []
    
    txt_files = [f for f in os.listdir(labels_dir) if f.endswith('.txt')]
    print(f"📄 Annotations trouvées: {len(txt_files)}")
    
    img_extensions = ('.jpg', '.jpeg', '.png', '.JPG', '.JPEG', '.PNG')
    
    for txt_file in txt_files:
        base_name = txt_file.replace('.txt', '')
        txt_path = os.path.join(labels_dir, txt_file)
        
        # Chercher l'image correspondante
        img_found = None
        train_root = os.path.join(base_path, 'train')
        
        for root, dirs, files in os.walk(train_root):
            for file in files:
                if file.lower().endswith(img_extensions):
                    if file.replace('.png', '').replace('.jpg', '').replace('.jpeg', '') == base_name:
                        img_found = os.path.join(root, file)
                        break
            if img_found:
                break
        
        if img_found:
            # Lire les classes
            classes = []
            with open(txt_path, 'r') as f:
                for line in f:
                    line = line.strip()
                    if line:
                        try:
                            class_id = int(line.split()[0])
                            classes.append(class_id)
                        except:
                            pass
            
            if classes:
                images_data.append({
                    'filename': os.path.basename(img_found),
                    'img_path': img_found,
                    'label_path': txt_path,
                    'classes': classes,
                    'primary_class': classes[0]
                })
    
    return images_data

print("\n📂 Chargement des images...")
all_images = load_all_images(DATASET_PATH)
print(f"✅ {len(all_images)} paires image+label trouvées")

# Split stratifié
print("\n📊 Création du split stratifié...")
img_paths = [img['img_path'] for img in all_images]
primary_classes = [img['primary_class'] for img in all_images]

train_paths, val_paths = train_test_split(
    img_paths,
    test_size=0.2,
    random_state=42,
    stratify=primary_classes
)

train_set = set(train_paths)
val_set = set(val_paths)

for img in all_images:
    if img['img_path'] in train_set:
        img['split'] = 'train'
    else:
        img['split'] = 'val'

# Compter par classe
train_counts = defaultdict(int)
val_counts = defaultdict(int)

for img in all_images:
    if img['split'] == 'train':
        train_counts[img['primary_class']] += 1
    else:
        val_counts[img['primary_class']] += 1

print("\n📊 Distribution par classe:")
print(" Classe            | Train |  Val  | Ratio Val")
print(" " + "-"*50)
for class_id, class_name in enumerate(CLASSES):
    train_c = train_counts.get(class_id, 0)
    val_c = val_counts.get(class_id, 0)
    total = train_c + val_c
    ratio = val_c / total if total > 0 else 0
    status = "✅" if 0.15 <= ratio <= 0.25 else "⚠️"
    print(f" {status} {class_name:17} | {train_c:5} | {val_c:5} | {ratio:.1%}")

# Créer les dossiers
print("\n📁 Création des dossiers...")
for split in ['train', 'val']:
    os.makedirs(os.path.join(OUTPUT_DIR, 'images', split), exist_ok=True)
    os.makedirs(os.path.join(OUTPUT_DIR, 'labels', split), exist_ok=True)

# Copier les fichiers
print("📁 Copie des fichiers...")
for img in all_images:
    split = img['split']
    dst_img = os.path.join(OUTPUT_DIR, 'images', split, img['filename'])
    if not os.path.exists(dst_img):
        shutil.copy2(img['img_path'], dst_img)
    
    dst_label = os.path.join(OUTPUT_DIR, 'labels', split, 
                             img['filename'].rsplit('.', 1)[0] + '.txt')
    if not os.path.exists(dst_label):
        shutil.copy2(img['label_path'], dst_label)

# Créer data.yaml
data_yaml = {
    'path': OUTPUT_DIR,
    'train': 'images/train',
    'val': 'images/val',
    'nc': len(CLASSES),
    'names': CLASSES
}

yaml_path = os.path.join(OUTPUT_DIR, 'data.yaml')
with open(yaml_path, 'w') as f:
    yaml.dump(data_yaml, f, default_flow_style=False)

print(f"\n✅ Dataset créé: {OUTPUT_DIR}")
print(f"✅ data.yaml: {yaml_path}")

# ============================================
# PARTIE 2 : ENTRAÎNEMENT YOLO11l
# ============================================

print("\n" + "="*70)
print("🚀 PARTIE 2 : ENTRAÎNEMENT YOLO11l")
print("="*70)

PROJECT_DIR = '/kaggle/working/runs_large'
NAME = 'maritime_yolo11l'
EPOCHS = 100
BATCH_SIZE = 6

print(f"\n📂 Dataset: {yaml_path}")
print(f"📁 Projet: {PROJECT_DIR}/{NAME}")
print(f"🔢 Epochs: {EPOCHS}")
print(f"📦 Batch size: {BATCH_SIZE}")

# Charger le modèle
print("\n🤖 Chargement de YOLO11l...")
model = YOLO('yolo11l.pt')

# Entraînement
print("\n📚 Début de l'entraînement...")
print("   (Cela prendra environ 3-4 heures sur Kaggle T4)")
print("="*70)

results = model.train(
    data=yaml_path,
    epochs=150,
    imgsz=640,
    batch=6,
    device=0,
    
    patience=50,
    dropout=0.2,
    
    augment=True,
    fliplr=0.3,
    degrees=5.0,
    mixup=0.05,
    copy_paste=0.0,
    
    lr0=0.01,
    weight_decay=0.0005,
    
    project=PROJECT_DIR,
    name=NAME,
    exist_ok=True,
    verbose=True
)

print("\n" + "="*70)
print("✅ ENTRAÎNEMENT TERMINÉ")
print("="*70)

# ============================================
# PARTIE 3 : ÉVALUATION ET RÉSULTATS
# ============================================

print("\n" + "="*70)
print("📊 PARTIE 3 : RÉSULTATS")
print("="*70)

# Lire les résultats
results_path = f'{PROJECT_DIR}/{NAME}/results.csv'
if os.path.exists(results_path):
    df = pd.read_csv(results_path)
    
    best_map95 = df['metrics/mAP50-95(B)'].max()
    best_map50 = df['metrics/mAP50(B)'].max()
    best_epoch = df['metrics/mAP50-95(B)'].idxmax()
    
    print(f"\n🏆 MEILLEURES PERFORMANCES (Validation):")
    print(f"   mAP50:     {best_map50:.4f}")
    print(f"   mAP50-95:  {best_map95:.4f}")
    print(f"   Epoch:     {best_epoch}")

# Évaluation sur validation
print("\n🔍 Évaluation finale sur validation...")
val_results = model.val(data=yaml_path, device=0, batch=8)

if hasattr(val_results, 'box'):
    print(f"\n📊 RÉSULTATS FINAUX VALIDATION:")
    print(f"   mAP50:     {val_results.box.map50:.4f}")
    print(f"   mAP50-95:  {val_results.box.map:.4f}")
    print(f"   Précision: {val_results.box.mp:.4f}")
    print(f"   Rappel:    {val_results.box.mr:.4f}")

# Comparaison avec YOLO11n
print("\n" + "="*70)
print("📈 COMPARAISON AVEC YOLO11n")
print("="*70)

print(f"\n YOLO11n (nano):    mAP50-95 = 0.326 (val), 0.281 (test)")
print(f" YOLO11l (large):   mAP50-95 = {best_map95:.4f} (val)")

# Sauvegarder le modèle
model_path = f'{PROJECT_DIR}/{NAME}/weights/best.pt'
print(f"\n✅ Modèle sauvegardé: {model_path}")
print(f"📁 Tous les résultats: {PROJECT_DIR}/{NAME}")

print("\n" + "="*70)
print("🎯 PROCHAINE ÉTAPE")
print("="*70)
print("\nUne fois l'entraînement terminé, nous pourrons :")
print("   1. Évaluer sur le test set avec XML")
print("   2. Comparer avec YOLO11n")
print("   3. Décider si besoin de YOLO26")
print("="*70)